# DeepSeek-V4-Flash-0731 vLLM sample tests

This notebook tests a running OpenAI-compatible vLLM service. By default it uses `http://127.0.0.1:8000`.

In [ ]:
import json
import os
from urllib.error import HTTPError, URLError
from urllib.request import Request, urlopen

BASE_URL = os.getenv("VLLM_BASE_URL", "http://127.0.0.1:8000").rstrip("/")
MODEL = os.getenv("VLLM_MODEL", "deepseek-ai/DeepSeek-V4-Flash-0731")
API_KEY = os.getenv("VLLM_API_KEY")

print("Service:", BASE_URL)
print("Model:", MODEL)

In [ ]:
def request_json(path, payload=None, timeout=180):
    headers = {"Content-Type": "application/json"}
    if API_KEY:
        headers["Authorization"] = f"Bearer {API_KEY}"
    body = None if payload is None else json.dumps(payload).encode("utf-8")
    request = Request(f"{BASE_URL}{path}", data=body, headers=headers)
    try:
        with urlopen(request, timeout=timeout) as response:
            content = response.read().decode("utf-8")
            return json.loads(content) if content else None
    except HTTPError as error:
        detail = error.read().decode("utf-8", errors="replace")
        raise RuntimeError(f"HTTP {error.code}: {detail}") from error
    except URLError as error:
        raise RuntimeError(f"Cannot reach {BASE_URL}: {error.reason}") from error


def chat(prompt, max_tokens=512, temperature=0):
    result = request_json(
        "/v1/chat/completions",
        {
            "model": MODEL,
            "messages": [{"role": "user", "content": prompt}],
            "max_tokens": max_tokens,
            "temperature": temperature,
        },
    )
    message = result["choices"][0]["message"]
    return {
        "reasoning": message.get("reasoning"),
        "content": message.get("content"),
        "usage": result.get("usage"),
        "fingerprint": result.get("system_fingerprint"),
    }

## Health and model discovery

In [ ]:
health = request_json("/health")
models = request_json("/v1/models")
assert any(item["id"] == MODEL for item in models["data"]), models
print("Health: OK")
print(json.dumps(models, indent=2))

## Basic generation

In [ ]:
result = chat(
    "Reply with one short sentence confirming that the service is working.",
    max_tokens=64,
)
print(result["content"])
print("Usage:", result["usage"])
print("Fingerprint:", result["fingerprint"])

## Algebra test

In [ ]:
result = chat(
    "Solve and briefly explain: If x + 1/x = 5, what is x^2 + 1/x^2?",
    max_tokens=256,
)
print(result["content"])
print("Expected answer: 23")

## Original AIME-style test

In [ ]:
result = chat(
    "Solve this original AIME-style problem. Give a clear derivation and a "
    "final three-digit answer: "
    "How many ordered 5-tuples of integers (x1,x2,x3,x4,x5) satisfy "
    "x1+x2+x3+x4+x5=12 and 0<=xi<=5 for every i?",
    max_tokens=1024,
)
print(result["content"])
print("Expected answer: 780")

## Ambiguity/reasoning test: Bertrand's paradox

A strong response should explain that the probability depends on how a random chord is selected. The common results are `1/4` for a uniformly distributed midpoint, `1/3` for two uniformly distributed endpoints, and `1/2` for a uniformly distributed distance from the center.

In [ ]:
result = chat(
    "Given a circle of radius 1, a chord is selected at random. What is the "
    "probability that the chord length exceeds sqrt(3)? Discuss any ambiguity "
    "in the question.",
    max_tokens=1024,
)
print(result["content"])
print("Reference values: midpoint=1/4, endpoints=1/3, center-distance=1/2")